# Single Task Fine-Tuning Visualization for GNN Models (TSMC)

This notebook visualizes how GNN models perform fine-tuning on a single random task.
Uses the same data format and validation logic as `TSMC_GCN_topology_validation.py`.

In [ ]:
import os
import sys
import torch
import torch.nn as nn
import numpy as np
import random
import matplotlib.pyplot as plt
from torch_geometric.data import Data, Batch

# Add paths
sys.path.append('../../../model_code/')
sys.path.append('../../../data_processing/gnn/')
sys.path.append('../utils/')

from gnn_maml import create_maml_gcn_model
from gnn_functions import evaluate_model_performance_gnn

In [ ]:
# ============================================================
# CONFIGURATION - Modify these settings as needed
# ============================================================

# Experiment type
EXPERIMENT = 'intra_topology'  # 'intra_topology' or 'topology_agnostic'

# Cell selection (set to None to use default cell list for experiment)
CELL_NAME = None  # e.g., 'AN4D0BWP30P140', 'ND3D0BWP30P140', or None for first available

# Data configuration
DATA_TYPE = 'cell'  # 'cell' or 'transition'
GRAPH_MODE = 'full_graph'  # 'stage_aware' or 'full_graph'
MODE = 'extrapolation'  # 'extrapolation' or 'interpolation'

# Model configuration
MODEL_TYPE = 'maml'  # 'baseline' or 'maml'
NUM_ITERATIONS = 100000

# MAML specific (only used if MODEL_TYPE == 'maml')
INNERDIV = 10
TASKS_PER_META_BATCH = 16
INNER_STEPS = 1

# Model architecture
CONV_HIDDEN_DIM = 128
NUM_CONV_LAYERS = 3
FC_HIDDEN_DIM = 128
NUM_FC_LAYERS = 2

# Paths
DATASET_DIR = '/home/tkdgn2907/Deepsets_test/MAML/Projects/dataset_all/dataset_TSMC_GNN_unified'
CACHE_DIR = '/home/tkdgn2907/Deepsets_test/MAML/Projects/data_processing/gnn/topology_cache'

GPU_ID = '0'

# Task selection
RANDOM_TASK_ID = None  # Set to None for random selection, or specify an integer
TOTAL_POINTS = 61

# Comparison mode (for MAML only)
COMPARE_MODE = False  # Set to True to compare 2 models
NUM_ITERATIONS_1 = 50000  # First model iteration count
NUM_ITERATIONS_2 = 100000  # Second model iteration count

In [ ]:
# GPU settings
os.environ["CUDA_VISIBLE_DEVICES"] = GPU_ID
os.environ["CUDA_DEVICE_ORDER"] = "PCI_BUS_ID"

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print('Device:', device)
if torch.cuda.is_available():
    print('Current cuda device:', torch.cuda.current_device())
    print('Count of using GPUs:', torch.cuda.device_count())

In [ ]:
# Cell lists for each experiment type (matching TSMC_GCN_topology_validation.py)
INTRA_TOPOLOGY_CELLS = [
    'AN4D0BWP30P140',
    'ND3D0BWP30P140',
    'NR3D1BWP30P140',
    'OR4D0BWP30P140',
    'XNR3D1BWP30P140',
    'XOR3D1BWP30P140',
]

TOPOLOGY_AGNOSTIC_CELLS = [
    'HA1D0BWP30P140', 'FA1D0BWP30P140', 'IOA21D0BWP30P140', 'IOA21D1BWP30P140',
    'OA21D0BWP30P140', 'OA21D1BWP30P140', 'OA211D0BWP30P140', 'OA211D1BWP30P140',
    'IAO21D0BWP30P140', 'IAO21D1BWP30P140', 'AO21D0BWP30P140', 'AO21D1BWP30P140',
    'AO211D0BWP30P140', 'AO211D1BWP30P140', 'SDFSNQD0BWP30P140', 'DFCNQD1BWP30P140'
]

# Select cell list based on experiment
if CELL_NAME is not None:
    cell_list = [CELL_NAME]
elif EXPERIMENT == 'intra_topology':
    cell_list = INTRA_TOPOLOGY_CELLS
else:
    cell_list = TOPOLOGY_AGNOSTIC_CELLS

print(f"Experiment: {EXPERIMENT}")
print(f"Available cells: {cell_list}")

In [ ]:
def normalize_node_features(node_features, norm_stats):
    """
    Normalize node features using saved statistics.
    Only normalize voltage (col 4), input_slew (col 5), output_load (col 6), temperature (col 10 for 11D)
    """
    if norm_stats is None:
        return node_features

    normalized = node_features.clone()

    # Normalize voltage (column 4)
    voltage_mask = normalized[:, 4] != 0
    if voltage_mask.any():
        normalized[voltage_mask, 4] = (
            normalized[voltage_mask, 4] - norm_stats['node_features']['voltage']['mean']
        ) / norm_stats['node_features']['voltage']['std']

    # Normalize input_slew (column 5)
    slew_mask = normalized[:, 5] != 0
    if slew_mask.any():
        normalized[slew_mask, 5] = (
            normalized[slew_mask, 5] - norm_stats['node_features']['input_slew']['mean']
        ) / norm_stats['node_features']['input_slew']['std']

    # Normalize output_load (column 6)
    load_mask = normalized[:, 6] != 0
    if load_mask.any():
        normalized[load_mask, 6] = (
            normalized[load_mask, 6] - norm_stats['node_features']['output_load']['mean']
        ) / norm_stats['node_features']['output_load']['std']

    # Normalize temperature (column 10 for 11D features) if available
    if 'temperature' in norm_stats['node_features'] and normalized.shape[1] > 10:
        temp_mask = normalized[:, 10] != 0
        if temp_mask.any():
            normalized[temp_mask, 10] = (
                normalized[temp_mask, 10] - norm_stats['node_features']['temperature']['mean']
            ) / norm_stats['node_features']['temperature']['std']

    return normalized


class CellTestDataset:
    """
    Dataset class for loading per-cell test data (same as validation script).
    """
    def __init__(self, cell_path, topology_cache=None):
        self.cell_path = cell_path
        self.topology_cache = topology_cache
        self._load_data()

    def _load_data(self):
        """Load cell test data"""
        data = torch.load(self.cell_path, weights_only=False, map_location='cpu')

        self._node_features = data['node_features']  # [num_libs, total_nodes, num_features]
        self._outputs = data['outputs']  # [num_libs, num_tasks]
        self._node_slices = data['node_slices']  # [num_tasks + 1]
        self._cell_name = data['cell_name']
        self._delay_types = data.get('delay_types', None)  # [num_tasks]
        self._output_names = data.get('output_names', None)  # [num_tasks]

        self.num_libs = data['num_libs']
        self.num_tasks = data['num_tasks']
        self.total_nodes = data['total_nodes']
        self.cell_name = data['cell_name']

    def get_task_data(self, task_idx, lib_idx, clone=True):
        """Get data for a specific task and lib."""
        node_start = self._node_slices[task_idx].item()
        node_end = self._node_slices[task_idx + 1].item()

        node_features = self._node_features[lib_idx, node_start:node_end, :]
        if clone:
            node_features = node_features.clone()

        output = self._outputs[lib_idx, task_idx].item()

        # Get delay_type and output_name for this task
        delay_type = 'rise'  # Default
        output_name = ''  # Default
        if self._delay_types is not None:
            delay_type = self._delay_types[task_idx]
        if self._output_names is not None:
            output_name = self._output_names[task_idx]

        return {
            'node_features': node_features,
            'output': output,
            'cell_name': self._cell_name,
            'delay_type': delay_type,
            'output_name': output_name,
        }

    def get_all_libs_for_task(self, task_idx, clone=True):
        """Get data for all libs for a specific task."""
        samples = []
        outputs = []
        for lib_idx in range(self.num_libs):
            sample = self.get_task_data(task_idx, lib_idx, clone=clone)
            samples.append(sample)
            outputs.append(sample['output'])
        return samples, outputs

    def get_task_outputs(self, task_idx):
        """Get all lib outputs for a specific task."""
        return self._outputs[:, task_idx]

In [ ]:
# Load train data for norm_stats and cache_path
train_path = os.path.join(DATASET_DIR, f"train_{DATA_TYPE}_{GRAPH_MODE}.pth")
print(f"Loading train data for norm_stats: {train_path}")

if not os.path.exists(train_path):
    raise FileNotFoundError(f"Train data not found: {train_path}")

train_data = torch.load(train_path, weights_only=False, map_location='cpu')
norm_stats = train_data.get('norm_stats', None)
cache_path = train_data.get('cache_path', None)

print(f"Norm stats loaded: {norm_stats is not None}")

# Load topology cache
if cache_path:
    if cache_path.startswith('/mnt/home/'):
        cache_path = cache_path.replace('/mnt/home/', '/home/')
    if not os.path.exists(cache_path):
        cache_filename = os.path.basename(cache_path)
        cache_path = os.path.join(CACHE_DIR, cache_filename)

if cache_path and os.path.exists(cache_path):
    print(f"Loading topology cache: {cache_path}")
    topology_cache = torch.load(cache_path, weights_only=False, map_location='cpu')
    print(f"Loaded topology cache for {len(topology_cache)} cells")
else:
    raise FileNotFoundError(f"Topology cache not found: {cache_path}")

In [ ]:
# Load cell test data
selected_cell = cell_list[0]  # Use first cell in list
print(f"\nSelected cell: {selected_cell}")

cell_path = os.path.join(
    DATASET_DIR,
    f"test_by_cell_{GRAPH_MODE}",
    f"{selected_cell}.pth"
)

if not os.path.exists(cell_path):
    raise FileNotFoundError(f"Cell data not found: {cell_path}")

cell_dataset = CellTestDataset(cell_path, topology_cache)
print(f"Loaded: {cell_dataset.num_tasks} tasks, {cell_dataset.num_libs} libs")

In [ ]:
# Load model(s)
def load_gnn_model(num_iterations, model_type='baseline'):
    """Load a GNN model checkpoint."""
    arch_suffix = f"_conv{CONV_HIDDEN_DIM}x{NUM_CONV_LAYERS}_fc{FC_HIDDEN_DIM}x{NUM_FC_LAYERS}"
    
    if model_type == 'baseline':
        model_dir = "../../../pretrained_models/gnn_baseline_checkpoints"
        model_filename = f"gnn_baseline_tsmc_process_{DATA_TYPE}_{GRAPH_MODE}_iter{num_iterations}{arch_suffix}.pth"
    else:  # maml
        model_dir = "../../../pretrained_models/gnn_maml_tsmc_process_final"
        model_filename = f"gnn_maml_tsmc_process_{DATA_TYPE}_{GRAPH_MODE}_innerdiv{INNERDIV}_meta{TASKS_PER_META_BATCH}_iter{num_iterations}_inner{INNER_STEPS}{arch_suffix}.pth"
    
    model_path = os.path.join(model_dir, model_filename)
    
    if not os.path.exists(model_path):
        raise FileNotFoundError(f"Model not found: {model_path}")
    
    checkpoint = torch.load(model_path, weights_only=False, map_location=device)
    
    # Get architecture params from checkpoint
    config = checkpoint.get('config', {})
    conv_hidden_dim = config.get('conv_hidden_dim', CONV_HIDDEN_DIM)
    num_conv_layers = config.get('num_conv_layers', NUM_CONV_LAYERS)
    fc_hidden_dim = config.get('fc_hidden_dim', FC_HIDDEN_DIM)
    num_fc_layers = config.get('num_fc_layers', NUM_FC_LAYERS)
    
    # Get node_features from checkpoint weight shape
    node_features = checkpoint['model_state_dict']['convs.0.lin.weight'].shape[1]
    
    # Create model
    model = create_maml_gcn_model(
        node_features=node_features,
        pooling='mean',
        output_dim=1,
        dropout=0.0,
        conv_hidden_dim=conv_hidden_dim,
        num_conv_layers=num_conv_layers,
        fc_hidden_dim=fc_hidden_dim,
        num_fc_layers=num_fc_layers
    )
    model.load_state_dict(checkpoint['model_state_dict'])
    model.eval()
    model = model.to(device)
    
    # Use norm_stats from checkpoint if available
    checkpoint_norm_stats = checkpoint.get('norm_stats', None)
    
    return model, model_path, checkpoint_norm_stats


print(f"\nLoading pretrained {MODEL_TYPE.upper()} model(s)...")

if COMPARE_MODE and MODEL_TYPE == 'maml':
    model1, model_path1, checkpoint_norm_stats = load_gnn_model(NUM_ITERATIONS_1, MODEL_TYPE)
    model2, model_path2, _ = load_gnn_model(NUM_ITERATIONS_2, MODEL_TYPE)
    
    print(f"Loaded Model 1 ({NUM_ITERATIONS_1} iters): {model_path1}")
    print(f"Loaded Model 2 ({NUM_ITERATIONS_2} iters): {model_path2}")
else:
    model, model_path, checkpoint_norm_stats = load_gnn_model(NUM_ITERATIONS, MODEL_TYPE)
    print(f"Loaded model: {model_path}")

# Use checkpoint norm_stats if available
if checkpoint_norm_stats is not None:
    norm_stats = checkpoint_norm_stats
    print("Using norm_stats from checkpoint")

In [ ]:
# Set mode-dependent indices
if MODE == 'extrapolation':
    indices = [5, 30, 55]
else:  # interpolation
    indices = [0, 13, 30, 45, 60]

# For interpolation mode: add endpoints
if MODE == 'interpolation':
    middle_indices = sorted(set(indices))
    if 0 not in middle_indices:
        middle_indices = [0] + middle_indices
    if TOTAL_POINTS - 1 not in middle_indices:
        middle_indices = middle_indices + [TOTAL_POINTS - 1]
    indices = middle_indices

k = len(indices)
left_bound = min(indices)
right_bound = max(indices) + 1
middle_idx = len(indices) // 2

print(f"Mode: {MODE}")
print(f"Indices: {indices}")
print(f"Left bound: {left_bound}, Right bound: {right_bound}")
print(f"K (support set size): {k}")

In [ ]:
# Select a random task
if RANDOM_TASK_ID is None:
    randomtask = random.randint(0, cell_dataset.num_tasks - 1)
else:
    randomtask = RANDOM_TASK_ID

print(f"\n" + "="*80)
print(f"Selected Task: {randomtask} / {cell_dataset.num_tasks}")
print(f"Cell: {selected_cell}")
print(f"Experiment: {EXPERIMENT}")
print(f"Model: {MODEL_TYPE}, Mode: {MODE}")
print(f"Support set indices: {indices}")
if COMPARE_MODE and MODEL_TYPE == 'maml':
    print(f"Comparison Mode: ON")
    print(f"Model 1 iterations: {NUM_ITERATIONS_1}")
    print(f"Model 2 iterations: {NUM_ITERATIONS_2}")
print("="*80)

# Load task data
task_samples, task_outputs = cell_dataset.get_all_libs_for_task(randomtask, clone=True)
task_outputs_tensor = torch.tensor(task_outputs, dtype=torch.float32)

# Apply normalization to all samples
for sample in task_samples:
    sample['node_features'] = normalize_node_features(sample['node_features'], norm_stats)

# Get support set
X_samples = [task_samples[idx] for idx in indices]
y = task_outputs_tensor[indices]

# Define regions
true_samples = task_samples
true_function = task_outputs_tensor
testdata_inter_output = task_outputs_tensor[left_bound:right_bound]
y_inter_mean = testdata_inter_output.mean()
y1_mean = task_outputs_tensor.mean()

if MODE == 'extrapolation':
    testdata_rightex_output = task_outputs_tensor[right_bound:]
    testdata_leftex_output = task_outputs_tensor[:left_bound]
    y_leftex_mean = testdata_leftex_output.mean()
    y_rightex_mean = testdata_rightex_output.mean()
else:
    y_leftex_mean = torch.tensor(0.0)
    y_rightex_mean = torch.tensor(0.0)

y_mean = y.mean()
y_std = y.std()

print(f"\nTask Statistics:")
print(f"  Support set mean: {y_mean.item():.6f}")
print(f"  Support set std: {y_std.item():.6f}")
print(f"  Full task mean: {y1_mean.item():.6f}")

# Print sample info
first_sample = task_samples[0]
print(f"\nSample Information:")
print(f"  Cell name: {first_sample['cell_name']}")
print(f"  Delay type: {first_sample['delay_type']}")
if first_sample['output_name']:
    print(f"  Output name: {first_sample['output_name']}")
print(f"  Node features shape: {first_sample['node_features'].shape}")

In [ ]:
# Evaluate model performance
if y_std > 0:
    y_norm = (y - y_mean) / y_std

    # Determine which model(s) to use
    models_to_eval = []
    if COMPARE_MODE and MODEL_TYPE == 'maml':
        models_to_eval = [
            (model1, f"MAML ({NUM_ITERATIONS_1} iters)", NUM_ITERATIONS_1),
            (model2, f"MAML ({NUM_ITERATIONS_2} iters)", NUM_ITERATIONS_2)
        ]
    else:
        models_to_eval = [(model, MODEL_TYPE.upper(), None)]

    # Create center input
    center_sample = task_samples[indices[0]]
    center_node_features = center_sample['node_features'].clone()
    center_node_features[:, 4] = 0.0  # Set voltage to 0 (normalized mean)

    # Get adjacency matrix from topology cache
    cell_cache = topology_cache[selected_cell]

    if GRAPH_MODE == 'stage_aware':
        output_name = center_sample.get('output_name', '')
        delay_type = center_sample.get('delay_type', 'rise')

        if not output_name and 'output_topologies' in cell_cache:
            available_outputs = list(cell_cache['output_topologies'].keys())
            if available_outputs:
                output_name = available_outputs[0]

        if 'output_topologies' in cell_cache and output_name in cell_cache['output_topologies']:
            output_topo = cell_cache['output_topologies'][output_name]
            if 'rise' in delay_type:
                adjacency_matrix = output_topo['pull_up']['adjacency_matrix']
            else:
                adjacency_matrix = output_topo['pull_down']['adjacency_matrix']
        else:
            adjacency_matrix = cell_cache.get('adjacency_matrix', torch.zeros((1, 1)))
    else:
        adjacency_matrix = cell_cache['adjacency_matrix']

    edge_index = adjacency_matrix.nonzero().t()
    center_data = Data(x=center_node_features, edge_index=edge_index)
    center_batch = Batch.from_data_list([center_data]).to(device)

    with torch.no_grad():
        center = models_to_eval[0][0](center_batch).item()

    y_max = y_norm.max().item()
    y_min = y_norm.min().item()

    # Get model predictions for scaling
    inter_predictions = []
    for idx in range(left_bound, right_bound):
        sample = task_samples[idx]
        data = Data(x=sample['node_features'], edge_index=edge_index)
        batch = Batch.from_data_list([data]).to(device)

        with torch.no_grad():
            pred = models_to_eval[0][0](batch).item()
            inter_predictions.append(pred)

    inter_predictions_tensor = torch.tensor(inter_predictions)
    min_val = inter_predictions_tensor.min().item()
    max_val = inter_predictions_tensor.max().item()

    if abs(max_val - min_val) > 1e-8:
        grad = (y_max - y_min) / (max_val - min_val)
        y_norm_middle = y_norm[middle_idx].item()
        move = center - y_norm_middle / grad

        print(f"\nScaling Parameters:")
        print(f"  Gradient: {grad:.6f}")
        print(f"  Move: {move:.6f}")
        print(f"  Center: {center:.6f}")

        # Store results for all models
        results = []

        for eval_model, eval_name, eval_iters in models_to_eval:
            print(f"\n{'='*80}")
            print(f"Evaluating: {eval_name}")
            print(f"{'='*80}")

            # Run model evaluation
            (total_loss1, inter_loss1, leftex_loss1, rightex_loss1,
             mape_loss1, leftex_mape1, inter_mape1, rightex_mape1,
             predictions, actual_values, output_min, _, mean_min, std_min, move_val, losses_min, adam_used,
             mae_loss1, leftex_mae1, inter_mae1, rightex_mae1) = evaluate_model_performance_gnn(
                eval_model, eval_name, X_samples, y,
                true_samples, true_function, grad, move,
                topology_cache, GRAPH_MODE, norm_stats, normalize_node_features,
                left_bound=left_bound, right_bound=right_bound, total_points=TOTAL_POINTS,
                mode=MODE
            )

            # Store results
            results.append({
                'name': eval_name,
                'iters': eval_iters,
                'total_loss': total_loss1,
                'inter_loss': inter_loss1,
                'leftex_loss': leftex_loss1,
                'rightex_loss': rightex_loss1,
                'mape': mape_loss1,
                'leftex_mape': leftex_mape1,
                'inter_mape': inter_mape1,
                'rightex_mape': rightex_mape1,
                'predictions': predictions,
                'actual_values': actual_values,
                'losses': losses_min,
                'adam_used': adam_used,
                'mae': mae_loss1,
                'leftex_mae': leftex_mae1,
                'inter_mae': inter_mae1,
                'rightex_mae': rightex_mae1,
                'mean': mean_min,
                'std': std_min
            })

            # Print results
            nrmse1 = (total_loss1 ** 0.5) / (y1_mean + 1e-8) * 100
            nrmse_inter = (inter_loss1 ** 0.5) / (y_inter_mean + 1e-8) * 100

            print(f"\nNRMSE (Normalized Root Mean Square Error):")
            print(f"  Total:               {nrmse1.item():.3f}%")
            print(f"  Interpolation:       {nrmse_inter.item():.3f}%")

            if MODE == 'extrapolation':
                nrmse_leftex = (leftex_loss1 ** 0.5) / (y_leftex_mean + 1e-8) * 100
                nrmse_rightex = (rightex_loss1 ** 0.5) / (y_rightex_mean + 1e-8) * 100
                print(f"  Left Extrapolation:  {nrmse_leftex.item():.3f}%")
                print(f"  Right Extrapolation: {nrmse_rightex.item():.3f}%")

            print(f"\nMAPE (Mean Absolute Percentage Error):")
            print(f"  Total:               {mape_loss1 * 100:.3f}%")
            print(f"  Interpolation:       {inter_mape1 * 100:.3f}%")

            if MODE == 'extrapolation':
                print(f"  Left Extrapolation:  {leftex_mape1 * 100:.3f}%")
                print(f"  Right Extrapolation: {rightex_mape1 * 100:.3f}%")

            print(f"\nMAE (Mean Absolute Error):")
            print(f"  Total:               {mae_loss1 * 1e9:.4f} ns")
            print(f"  Interpolation:       {inter_mae1 * 1e9:.4f} ns")

            if MODE == 'extrapolation':
                print(f"  Left Extrapolation:  {leftex_mae1 * 1e9:.4f} ns")
                print(f"  Right Extrapolation: {rightex_mae1 * 1e9:.4f} ns")

            print(f"\nAdam condition triggered: {adam_used}")
    else:
        print("\nError: max_val == min_val, cannot compute gradient")
        results = []
else:
    print("\nError: y_std is zero")
    results = []

In [ ]:
# Visualization: Prediction vs Actual scatter plot
if len(results) > 0:
    result = results[0]
    predictions = result['predictions']
    actual_values = result['actual_values']

    plt.figure(figsize=(8, 8))
    plt.scatter(actual_values, predictions, alpha=0.5)

    # Plot diagonal line (perfect prediction)
    min_val = min(min(actual_values), min(predictions))
    max_val = max(max(actual_values), max(predictions))
    plt.plot([min_val, max_val], [min_val, max_val], 'r--', linewidth=2, label='Perfect prediction')

    plt.xlabel('Actual Delay (ns)', fontsize=12)
    plt.ylabel('Predicted Delay (ns)', fontsize=12)
    plt.title(f'Prediction vs Actual - {result["name"]}\nCell: {selected_cell}, Task: {randomtask}', fontsize=14)
    plt.legend()
    plt.grid(True, alpha=0.3)
    plt.axis('equal')
    plt.tight_layout()
    plt.show()

In [ ]:
# Visualization: Loss curve and prediction plot
if len(results) > 0:
    result = results[0]

    fig, axes = plt.subplots(1, 2, figsize=(16, 5))

    # Plot 1: Predictions vs actual
    ax = axes[0]
    x_axis = np.arange(TOTAL_POINTS)

    ax.plot(x_axis, result['actual_values'], 'o-', label='Ground Truth', alpha=0.7, markersize=4)
    ax.plot(x_axis, result['predictions'], 's-', label='Predictions', alpha=0.7, markersize=4)

    # Mark support set points
    support_y = [result['actual_values'][i] for i in indices]
    ax.scatter(indices, support_y, color='red', s=100, zorder=5, marker='x', label='Support Set')

    # Mark regions
    if MODE == 'extrapolation':
        ax.axvspan(0, left_bound, alpha=0.1, color='blue', label='Left Extrapolation')
        ax.axvspan(left_bound, right_bound, alpha=0.1, color='green', label='Interpolation')
        ax.axvspan(right_bound, TOTAL_POINTS, alpha=0.1, color='orange', label='Right Extrapolation')
    else:
        ax.axvspan(left_bound, right_bound, alpha=0.1, color='green', label='Interpolation')

    ax.set_xlabel('Sample Index (Voltage)', fontsize=12)
    ax.set_ylabel('Delay (ns)', fontsize=12)
    ax.set_title(f'Model Predictions - {result["name"]}\nCell: {selected_cell}, Task: {randomtask}', fontsize=14, fontweight='bold')
    ax.legend(loc='best')
    ax.grid(True, alpha=0.3)

    # Plot 2: Loss curve
    ax = axes[1]
    losses = result['losses']

    if len(losses) > 5:
        ax.plot(losses[5:], linewidth=2)
        ax.set_xlabel('Gradient steps (after first 5)', fontsize=12)
    else:
        ax.plot(losses, linewidth=2)
        ax.set_xlabel('Gradient steps', fontsize=12)

    ax.set_ylabel('MSE Loss', fontsize=12)
    ax.set_title('Fine-tuning Loss Curve', fontsize=14, fontweight='bold')
    ax.grid(True, alpha=0.3)

    plt.tight_layout()
    plt.show()

In [ ]:
# Comparison visualization (if comparing 2 models)
if COMPARE_MODE and MODEL_TYPE == 'maml' and len(results) == 2:
    print(f"\n{'='*80}")
    print("COMPARISON SUMMARY")
    print(f"{'='*80}\n")

    # Create comparison table
    print(f"{'Metric':<30} | {results[0]['name']:<25} | {results[1]['name']:<25} | Delta")
    print("-" * 100)

    # NRMSE comparisons
    nrmse1_total = (results[0]['total_loss'] ** 0.5) / (y1_mean + 1e-8) * 100
    nrmse2_total = (results[1]['total_loss'] ** 0.5) / (y1_mean + 1e-8) * 100
    delta_nrmse = nrmse1_total.item() - nrmse2_total.item()

    print(f"{'NRMSE - Total (%)':<30} | {nrmse1_total.item():<25.3f} | {nrmse2_total.item():<25.3f} | {delta_nrmse:+.3f}")

    nrmse1_inter = (results[0]['inter_loss'] ** 0.5) / (y_inter_mean + 1e-8) * 100
    nrmse2_inter = (results[1]['inter_loss'] ** 0.5) / (y_inter_mean + 1e-8) * 100
    delta_nrmse_inter = nrmse1_inter.item() - nrmse2_inter.item()

    print(f"{'NRMSE - Interpolation (%)':<30} | {nrmse1_inter.item():<25.3f} | {nrmse2_inter.item():<25.3f} | {delta_nrmse_inter:+.3f}")

    if MODE == 'extrapolation':
        nrmse1_left = (results[0]['leftex_loss'] ** 0.5) / (y_leftex_mean + 1e-8) * 100
        nrmse2_left = (results[1]['leftex_loss'] ** 0.5) / (y_leftex_mean + 1e-8) * 100
        delta_nrmse_left = nrmse1_left.item() - nrmse2_left.item()

        nrmse1_right = (results[0]['rightex_loss'] ** 0.5) / (y_rightex_mean + 1e-8) * 100
        nrmse2_right = (results[1]['rightex_loss'] ** 0.5) / (y_rightex_mean + 1e-8) * 100
        delta_nrmse_right = nrmse1_right.item() - nrmse2_right.item()

        print(f"{'NRMSE - Left Extrap. (%)':<30} | {nrmse1_left.item():<25.3f} | {nrmse2_left.item():<25.3f} | {delta_nrmse_left:+.3f}")
        print(f"{'NRMSE - Right Extrap. (%)':<30} | {nrmse1_right.item():<25.3f} | {nrmse2_right.item():<25.3f} | {delta_nrmse_right:+.3f}")

    print("\nNote: Positive Delta means Model 2 performs better")
    print("      Negative Delta means Model 1 performs better")

    # Overlaid comparison plot
    fig, axes = plt.subplots(1, 2, figsize=(16, 5))

    # Plot 1: Overlaid predictions
    ax = axes[0]
    x_axis = np.arange(TOTAL_POINTS)

    ax.plot(x_axis, results[0]['actual_values'], 'o-', label='Ground Truth',
            alpha=0.5, markersize=3, color='gray')
    ax.plot(x_axis, results[0]['predictions'], 's-', label=results[0]['name'],
            alpha=0.7, markersize=4)
    ax.plot(x_axis, results[1]['predictions'], '^-', label=results[1]['name'],
            alpha=0.7, markersize=4)

    # Mark support set
    support_y = [results[0]['actual_values'][i] for i in indices]
    ax.scatter(indices, support_y, color='red', s=100, zorder=5, marker='x', label='Support Set')

    ax.set_xlabel('Sample Index (Voltage)', fontsize=12)
    ax.set_ylabel('Delay (ns)', fontsize=12)
    ax.set_title(f'Comparison: Model Predictions\nCell: {selected_cell}, Task: {randomtask}', fontsize=14, fontweight='bold')
    ax.legend(loc='best')
    ax.grid(True, alpha=0.3)

    # Plot 2: Overlaid loss curves
    ax = axes[1]

    for idx, result in enumerate(results):
        losses = result['losses']
        plot_losses = losses[5:] if len(losses) > 5 else losses
        ax.plot(plot_losses, linewidth=2, label=result['name'])

    ax.set_xlabel('Gradient steps (after first 5)', fontsize=12)
    ax.set_ylabel('MSE Loss', fontsize=12)
    ax.set_title('Comparison: Fine-tuning Loss Curves', fontsize=14, fontweight='bold')
    ax.legend()
    ax.grid(True, alpha=0.3)

    plt.tight_layout()
    plt.show()

In [ ]:
# Additional analysis: Show per-region errors
if len(results) > 0:
    result = results[0]
    predictions = np.array(result['predictions'])
    actuals = np.array(result['actual_values'])
    
    # Calculate errors per region
    errors = np.abs(predictions - actuals)
    
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    
    # Plot 1: Absolute error per sample
    ax = axes[0]
    x_axis = np.arange(TOTAL_POINTS)
    ax.bar(x_axis, errors * 1e9, alpha=0.7, color='steelblue')
    
    if MODE == 'extrapolation':
        ax.axvspan(0, left_bound, alpha=0.1, color='blue', label='Left Extrapolation')
        ax.axvspan(left_bound, right_bound, alpha=0.1, color='green', label='Interpolation')
        ax.axvspan(right_bound, TOTAL_POINTS, alpha=0.1, color='orange', label='Right Extrapolation')
    
    ax.set_xlabel('Sample Index (Voltage)', fontsize=12)
    ax.set_ylabel('Absolute Error (ns)', fontsize=12)
    ax.set_title('Absolute Error per Sample', fontsize=14, fontweight='bold')
    ax.legend()
    ax.grid(True, alpha=0.3, axis='y')
    
    # Plot 2: Error distribution
    ax = axes[1]
    
    if MODE == 'extrapolation':
        left_errors = errors[:left_bound] * 1e9
        inter_errors = errors[left_bound:right_bound] * 1e9
        right_errors = errors[right_bound:] * 1e9
        
        data = [left_errors, inter_errors, right_errors]
        labels = ['Left Extrap', 'Interpolation', 'Right Extrap']
    else:
        data = [errors * 1e9]
        labels = ['All']
    
    bp = ax.boxplot(data, labels=labels, patch_artist=True)
    colors = ['lightblue', 'lightgreen', 'lightyellow'][:len(data)]
    for patch, color in zip(bp['boxes'], colors):
        patch.set_facecolor(color)
    
    ax.set_ylabel('Absolute Error (ns)', fontsize=12)
    ax.set_title('Error Distribution by Region', fontsize=14, fontweight='bold')
    ax.grid(True, alpha=0.3, axis='y')
    
    plt.tight_layout()
    plt.show()
    
    # Print statistics
    print(f"\nError Statistics:")
    print(f"  Mean Error: {errors.mean() * 1e9:.4f} ns")
    print(f"  Max Error:  {errors.max() * 1e9:.4f} ns")
    print(f"  Min Error:  {errors.min() * 1e9:.4f} ns")
    
    if MODE == 'extrapolation':
        print(f"\n  Left Extrapolation Mean:  {left_errors.mean():.4f} ns")
        print(f"  Interpolation Mean:       {inter_errors.mean():.4f} ns")
        print(f"  Right Extrapolation Mean: {right_errors.mean():.4f} ns")

## Optimization Method Comparison

Compare 4 different optimization methods on the same task:
1. **Grad + Move only** - No optimization, just scaling
2. **SGD 50 iterations** - Standard SGD optimization
3. **Adam 50 iterations** - Adam optimization
4. **Current (Selective Adam)** - Grad + Move + Adam if loss > threshold

In [ ]:
# Import comparison functions
from gnn_functions import (
    compare_optimization_methods_gnn,
    plot_optimization_comparison_gnn,
    print_optimization_comparison_summary_gnn
)

# Run comparison on the same task
if y_std > 0:
    print(f"Running optimization method comparison on Task {randomtask}...")
    print(f"Methods: Grad+Move Only, SGD 50 steps, Adam 50 steps, Selective Adam")
    
    # Use the first model for comparison
    eval_model = model if not COMPARE_MODE else model1
    
    comparison_results = compare_optimization_methods_gnn(
        initial_model=eval_model,
        X_samples=X_samples,
        y=y,  # Original scale support set outputs
        true_samples=true_samples,
        true_function=true_function,
        grad=grad,
        move=move,
        topology_cache=topology_cache,
        cache_type=GRAPH_MODE,
        norm_stats=norm_stats,
        normalize_fn=normalize_node_features,
        num_steps=50,
        left_bound=left_bound,
        right_bound=right_bound,
        total_points=TOTAL_POINTS,
        mode=MODE
    )
    
    # Print summary
    print_optimization_comparison_summary_gnn(comparison_results, mode=MODE)
    
    # Plot comparison
    fig = plot_optimization_comparison_gnn(
        comparison_results,
        indices=indices,
        total_points=TOTAL_POINTS,
        left_bound=left_bound,
        right_bound=right_bound,
        mode=MODE,
        cell_name=selected_cell,
        task_id=randomtask
    )
    plt.show()
else:
    print("Cannot run comparison: y_std is zero")